JAX HOUSE PRICE PREDICTION

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

CASE STUDY 01

In [ ]:
import jax
import jax.numpy as jnp
from jax import grad, jit
from jax.lax import fori_loop
import pandas as pd
import time

# -------------------------
# Read Data
# -------------------------
data_path = "/kaggle/input/competitions/house-prices-advanced-regression-techniques/"
df = pd.read_csv(data_path + "train.csv")

# -------------------------
# Select Columns
# -------------------------
cols = ["GrLivArea", "BedroomAbvGr", "FullBath", "OverallQual", "YearBuilt"]

inputs = df[cols].fillna(0).values
targets = df["SalePrice"].values

inputs = jnp.array(inputs)
targets = jnp.array(targets)

# -------------------------
# Standardization
# -------------------------
mean_X = jnp.mean(inputs, axis=0)
std_X = jnp.std(inputs, axis=0) + 1e-8
inputs = (inputs - mean_X) / std_X

mean_y = jnp.mean(targets)
std_y = jnp.std(targets) + 1e-8
targets = (targets - mean_y) / std_y

# -------------------------
# Add Intercept
# -------------------------
inputs = jnp.concatenate([jnp.ones((inputs.shape[0], 1)), inputs], axis=1)

# -------------------------
# Prediction Function
# -------------------------
def forward(weights, inputs):
    return jnp.matmul(inputs, weights)

# -------------------------
# Error Function
# -------------------------
def mse_loss(weights, inputs, targets):
    output = forward(weights, inputs)
    return jnp.mean((output - targets) ** 2)

# -------------------------
# Auto Gradient
# -------------------------
gradient_fn = grad(mse_loss)

# -------------------------
# Parameter Update
# -------------------------
@jit
def step(weights, inputs, targets, alpha):
    grads = gradient_fn(weights, inputs, targets)
    return weights - alpha * grads

# -------------------------
# Loop for Training
# -------------------------
@jit
def run_training(weights, inputs, targets, alpha, iterations):
    def loop_body(i, w):
        return step(w, inputs, targets, alpha)
    return fori_loop(0, iterations, loop_body, weights)

# -------------------------
# Initialize Parameters
# -------------------------
key = jax.random.PRNGKey(42)
weights = jax.random.normal(key, (inputs.shape[1],))

# -------------------------
# Start Timing
# -------------------------
t1 = time.time()

# -------------------------
# Train
# -------------------------
weights = run_training(weights, inputs, targets, alpha=0.005, iterations=8000)

# -------------------------
# End Timing
# -------------------------
t2 = time.time()
total_time = t2 - t1

# -------------------------
# Predictions
# -------------------------
outputs = forward(weights, inputs)

# Back to original scale
outputs = outputs * std_y + mean_y
targets_real = targets * std_y + mean_y

# -------------------------
# Metrics
# -------------------------
error = jnp.mean((outputs - targets_real) ** 2)
rmse_val = jnp.sqrt(error)

# -------------------------
# Display Results
# -------------------------
print("Final MSE:", error)
print("Final RMSE:", rmse_val)
print("Execution Time:", total_time)
print("Sample Predictions:", outputs[:5])

In [ ]:
import time

start = time.time()

for i in range(500):
    w = w - 1e-7 * grad_fn(w, X, y)

end = time.time()

print("Time without jit:", end - start)

In [ ]:
start = time.time()

w = train_loop(w, X, y, lr=1e-7, steps=500)

end = time.time()

print("Time with jit:", end - start)

WITHOUT JAX

In [ ]:
import numpy as np
import pandas as pd
import time

# -------------------------
# Load Data
# -------------------------
data_dir = "/kaggle/input/competitions/house-prices-advanced-regression-techniques/"
df = pd.read_csv(data_dir + "train.csv")

# -------------------------
# Choose Features
# -------------------------
cols = ["GrLivArea", "BedroomAbvGr", "FullBath", "OverallQual", "YearBuilt"]

A = df[cols].fillna(0).to_numpy(dtype=np.float64)
b = df["SalePrice"].to_numpy(dtype=np.float64)

# -------------------------
# Scaling
# -------------------------
A_avg = np.mean(A, axis=0)
A_std = np.std(A, axis=0) + 1e-8
A = (A - A_avg) / A_std

b_avg = np.mean(b)
b_std = np.std(b) + 1e-8
b = (b - b_avg) / b_std

# -------------------------
# Add Intercept Term
# -------------------------
A = np.concatenate([np.ones((A.shape[0], 1)), A], axis=1)

# -------------------------
# Forward Pass
# -------------------------
def forward_pass(w, A):
    return A @ w

# -------------------------
# Loss Calculation
# -------------------------
def mse_cost(w, A, b):
    pred = forward_pass(w, A)
    return np.mean((pred - b) ** 2)

# -------------------------
# Manual Gradient
# -------------------------
def gradient_calc(w, A, b):
    pred = forward_pass(w, A)
    diff = pred - b
    return (2 / A.shape[0]) * (A.T @ diff)

# -------------------------
# Training Function
# -------------------------
def run_training(A, b, lr=0.005, epochs=8000):
    t_start = time.time()

    weights = np.random.randn(A.shape[1])

    for step in range(epochs):
        g = gradient_calc(weights, A, b)
        weights -= lr * g

        if step % 2000 == 0:
            current_loss = mse_cost(weights, A, b)
            print(f"Iteration {step} -> Loss: {current_loss}")

    t_end = time.time()
    return weights, (t_end - t_start)

# -------------------------
# Train Model
# -------------------------
weights, total_time = run_training(A, b)

# -------------------------
# Predictions
# -------------------------
outputs = forward_pass(weights, A)

# Back to original scale
outputs = outputs * b_std + b_avg
actual = b * b_std + b_avg

# -------------------------
# Evaluation
# -------------------------
final_mse = np.mean((outputs - actual) ** 2)
final_rmse = np.sqrt(final_mse)

# -------------------------
# Results
# -------------------------
print("\n===== NUMPY MODEL RESULTS =====")
print("MSE:", final_mse)
print("RMSE:", final_rmse)
print("Time Taken:", total_time)
print("Sample Outputs:", outputs[:5])